In [ ]:
import pandas as pd
import numpy as np
import random
import warnings

In [ ]:
# Suppress all FutureWarnings
warnings.simplefilter(action='ignore', category=FutureWarning)
price_data = pd.read_csv('E:\Signal Backtesting\Input\price_2023-06-01_to_2024-08-4_min.csv', parse_dates=['Datetime'],
                         index_col='Datetime')
# Re-load the signal data without setting the index
signal_data = pd.read_csv('E:\Signal Backtesting\Input\\filtered_signals_with_2023_2024_all_events_plus_one_hour.csv',
                            parse_dates=['Datetime'])

#month = 7  # January (you can change this to the desired month)
year = 2024  # You can change this to the desired year
# # Filter the signal data for the specified month and year
signal_data = signal_data[(signal_data['Datetime'].dt.year == year)]

In [ ]:
# Backtest trades function
def backtest_trades(price_data, signal_data, tp=None, sl=None, entry_time_offset=None,
                    percentage_change=None, open_order_elimination=None, ignore_time_interval_before=None,
                    ignore_time_interval_after=None):
    output_data = pd.DataFrame(columns=[
        'Datetime', 'Side', 'Signal Open Price', 'Entry Price', 'TP Price', 'SL Price', 'Result', 'Duration',
        'Execution Latency', 'ROI', 'NAV', 'Ignore Reason'
    ])

    initial_margin = 100000
    current_margin = initial_margin
    exit_datetimes = []
    initial_drawdown = 0  # For initial drawdown calculation
    nav_history = []
    
    # If signal_data is a Series (single signal), convert it to a DataFrame
    if isinstance(signal_data, pd.Series):
        signal_data = signal_data.to_frame().T
    
    for i, row in signal_data.iterrows():
        signal_datetime = row['Datetime']
        signal_value = row['Signal']

        if signal_value == 0:
            continue
        elif signal_value > 0:
            side = 'Buy'
        else:
            side = 'Sell'

        adjusted_signal_datetime = signal_datetime + pd.Timedelta(minutes=entry_time_offset)
        # Check if the adjusted datetime exists in the price data
        if adjusted_signal_datetime not in price_data.index:
         
          try:
            # Find the nearest timestamp
            closest_index = price_data.index.get_loc(adjusted_signal_datetime, method='nearest')
            closest_timestamp = price_data.index[closest_index]
            print(f"Adjusted timestamp {adjusted_signal_datetime} not found, using closest timestamp {closest_timestamp} instead.")
            adjusted_signal_datetime = closest_timestamp
          except KeyError:
            # If no close timestamp can be found, log and skip this signal
            print(f"Adjusted timestamp {adjusted_signal_datetime} could not be matched. Skipping this signal.")
            return None
        signal_open_price = price_data.at[adjusted_signal_datetime, 'Open']

        # Ignoring signals based on open trades
        exit_datetimes.sort(key=lambda x: x[0])
        ignore_signal = False
        reason = ''

        if exit_datetimes:
            later_exits = [ed for ed in exit_datetimes if ed[0] > signal_datetime]

            if len(later_exits) >= 2:
                result = 'Ignored'
                reason = '2 open trades'
                ignore_signal = True
            elif len(later_exits) == 1:
                if later_exits[-1][1] != side:
                    ignore_signal = False
                else:
                    result = 'Ignored'
                    reason = 'One open trade with the same side'
                    ignore_signal = True

        if ignore_signal:
            new_row = pd.DataFrame([{
                'Datetime': signal_datetime,
                'Side': side,
                'Signal Open Price': signal_open_price,
                'Entry Price': None,
                'TP Price': None,
                'SL Price': None,
                'Result': result,
                'Duration': '00:00:00',
                'Execution Latency': '00:00:00',
                'ROI': 0,
                'NAV': current_margin,
                'Ignore Reason': reason
            }])
            output_data = pd.concat([output_data, new_row], ignore_index=True)
            continue

        # New logic: ignore signals within a specific time interval before and after events
        if 'event_datetimes' in row and pd.notna(row['event_datetimes']):
            event_times = [pd.to_datetime(e.strip()) for e in str(row['event_datetimes']).split(',')]
            ignore_signal = any(event_datetime - pd.Timedelta(
                minutes=ignore_time_interval_before) <= signal_datetime <= event_datetime + pd.Timedelta(
                minutes=ignore_time_interval_after)
                                for event_datetime in event_times)
        if ignore_signal:
            new_row = pd.DataFrame([{
                'Datetime': signal_datetime,
                'Side': side,
                'Signal Open Price': signal_open_price,
                'Entry Price': None,
                'TP Price': None,
                'SL Price': None,
                'Result': 'Ignored',
                'Duration': '00:00:00',
                'Execution Latency': '00:00:00',
                'ROI': 0,
                'NAV': current_margin,
                'Ignore Reason': 'Signal around economic event'
            }])
            output_data = pd.concat([output_data, new_row], ignore_index=True)
            continue

        entry_datetime, entry_price, entry_duration = determine_entry(price_data, signal_datetime, percentage_change,
                                                                      side, open_order_elimination, entry_time_offset)
        if entry_datetime is None:
            new_row = pd.DataFrame([{
                'Datetime': signal_datetime,
                'Side': side,
                'Signal Open Price': signal_open_price,
                'Entry Price': None,
                'TP Price': None,
                'SL Price': None,
                'Result': 'Not Filled',
                'Duration': '00:00:00',
                'Execution Latency': '00:00:00',
                'ROI': 0,
                'NAV': current_margin,
                'Ignore Reason': ''
            }])
            output_data = pd.concat([output_data, new_row], ignore_index=True)
            continue

        # Update NAV on filled order
        current_margin *= (1 - 0.0002)

        if side == 'Buy':
            tp_price = entry_price * (1 + tp)
            sl_price = entry_price * (1 - sl)
        else:
            tp_price = entry_price * (1 - tp)
            sl_price = entry_price * (1 + sl)

        result, duration_str = check_tp_sl(price_data, entry_datetime, tp_price, sl_price, side)
        exit_datetime = entry_datetime + pd.Timedelta(duration_str)

        if result in [1, -1]:
            exit_datetimes.append((exit_datetime, side))

        if 'event_datetimes' in row and pd.notna(row['event_datetimes']):
            event_times = [pd.to_datetime(e.strip()) for e in str(row['event_datetimes']).split(',')]
            for event_datetime in event_times:
                if entry_datetime < event_datetime < exit_datetime:
                    exit_datetime = event_datetime - pd.Timedelta(minutes=10)
                    if exit_datetime in price_data.index:
                        exit_price = price_data.at[exit_datetime, 'Open']
                        result = 'ended before data'
                        if (side == 'Buy' and exit_price > entry_price) or (
                                side == 'Sell' and exit_price < entry_price):
                            result += ' with profit'
                            pct_change = (exit_price - entry_price) / entry_price if side == 'Buy' else (
                                                                                                                entry_price - exit_price) / entry_price
                            current_margin = current_margin * (1 + pct_change)
                            current_margin *= (1 - 0.0005)
                        else:
                            result += ' with loss'
                            pct_change = (entry_price - exit_price) / entry_price if side == 'Buy' else (
                                                                                                                exit_price - entry_price) / entry_price
                            current_margin = current_margin * (1 - pct_change)
                            current_margin *= (1 - 0.0005)
                    break

        if result not in ['ended before data with profit', 'ended before data with loss',
                          'ended before data with no exact price']:

            if result == 1:
                current_margin = current_margin * (1 + tp)
                current_margin *= (1 - 0.0005)
            elif result == -1:
                current_margin = current_margin * (1 - sl)
                current_margin *= (1 - 0.0005)

        # Calculate initial drawdown
        if current_margin < 100000:
            drawdown = ((100000 - current_margin) / 100000) * 100
            initial_drawdown = max(initial_drawdown, drawdown)

        roi = ((current_margin - initial_margin) / initial_margin) * 100
        nav = current_margin
        initial_margin = current_margin

        nav_history.append(nav)
        new_row = pd.DataFrame([{
            'Datetime': signal_datetime,
            'Side': side,
            'Signal Open Price': signal_open_price,
            'Entry Price': entry_price,
            'TP Price': tp_price,
            'SL Price': sl_price,
            'Result': result,
            'Duration': duration_str,
            'Execution Latency': format_duration(entry_duration),
            'ROI': roi,
            'NAV': nav,
            'Ignore Reason': ''
        }])

        output_data = pd.concat([output_data, new_row], ignore_index=True)

    # Ensure 'Datetime' column is in datetime format
    output_data['Datetime'] = pd.to_datetime(output_data['Datetime'])

    # Calculate Daily Return
    output_data['Date'] = output_data['Datetime'].dt.date
    daily_nav = output_data.groupby('Date')['NAV'].last().to_dict()
    daily_returns = {}
    previous_day_nav = 100000

    for date, nav in daily_nav.items():
        daily_return = ((nav - previous_day_nav) / previous_day_nav) * 100
        daily_returns[date] = daily_return
        previous_day_nav = nav

    output_data['Daily Return'] = output_data['Date'].map(daily_returns)
    output_data.drop(columns=['Date'], inplace=True)

    # Calculate Monthly Max Drawdown
    output_data['Month'] = output_data['Datetime'].dt.to_period('M')
    monthly_max_drawdowns = {}

    for month, group in output_data.groupby('Month'):
        peak_nav = group['NAV'].iloc[0]  # Start with the first NAV of the month
        max_drawdown_in_month = 0
        local_peak = peak_nav

        for nav in group['NAV']:
            if nav > local_peak:
                local_peak = nav  # Update the peak if a new high is found
            else:
                # Calculate drawdown from the peak to the current NAV
                drawdown = ((local_peak - nav) / local_peak) * 100
                max_drawdown_in_month = max(max_drawdown_in_month, drawdown)  # Track the maximum drawdown

        monthly_max_drawdowns[month] = max_drawdown_in_month

    output_data['Monthly Max Drawdown'] = output_data['Month'].map(monthly_max_drawdowns)
    output_data['Initial Drawdown'] = initial_drawdown

    output_data.drop(columns=['Month'], inplace=True)

    return output_data


# Helper functions
def format_duration(duration):
    seconds = duration.total_seconds()
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    seconds = int(seconds % 60)
    return f"{hours:02}:{minutes:02}:{seconds:02}"


def check_tp_sl(price_data, entry_datetime, tp_price, sl_price, side):
    result = 0
    exit_datetime = None
    subsequent_prices = price_data.loc[entry_datetime:]

    for current_datetime, price_row in subsequent_prices.iterrows():
        if side == 'Buy':
            if price_row['High'] >= tp_price:
                result = 1
                exit_datetime = current_datetime
                break
            elif price_row['Low'] <= sl_price:
                result = -1
                exit_datetime = current_datetime
                break
        else:
            if price_row['Low'] <= tp_price:
                result = 1
                exit_datetime = current_datetime
                break
            elif price_row['High'] >= sl_price:
                result = -1
                exit_datetime = current_datetime
                break

    if exit_datetime:
        duration = exit_datetime - entry_datetime
        duration_str = format_duration(duration)
    else:
        duration_str = '00:00:00'

    return result, duration_str


def determine_entry(price_data, signal_datetime, percentage_change, side, open_order_elimination, entry_time_offset):
    adjusted_signal_datetime = signal_datetime + pd.Timedelta(minutes=entry_time_offset)
    if adjusted_signal_datetime not in price_data.index:
        return None, None, None

    adjusted_open_price = price_data.at[adjusted_signal_datetime, 'Open']
    percentage_change_price = adjusted_open_price * (
                1 - percentage_change) if side == 'Buy' else adjusted_open_price * (1 + percentage_change)

    time_limit = adjusted_signal_datetime + pd.Timedelta(minutes=open_order_elimination)
    subsequent_prices = price_data.loc[adjusted_signal_datetime:time_limit]

    for current_datetime, price_row in subsequent_prices.iterrows():
        if side == 'Buy' and price_row['Low'] <= percentage_change_price:
            entry_price = percentage_change_price
            duration = current_datetime - adjusted_signal_datetime
            return current_datetime, entry_price, duration
        elif side == 'Sell' and price_row['High'] >= percentage_change_price:
            entry_price = percentage_change_price
            duration = current_datetime - adjusted_signal_datetime
            return current_datetime, entry_price, duration

    return None, None, None

In [ ]:
tp_values = np.arange(0.003, 0.014, 0.001)
sl_values = np.arange(0.003, 0.014, 0.001)
# entry_time_offset_values = np.arange(0, 120, 10)
percentage_change_values = np.arange(0.0001, 0.0015, 0.0001)

In [ ]:
def create_initial_solution():
    return {
        'tp': np.random.choice(tp_values),
        'sl': np.random.choice(sl_values),
        'percentage_change': np.random.choice(percentage_change_values)
    }

In [ ]:
def generate_neighbor(solution):
    new_solution = solution.copy()
    mutation_type = random.choice(['tp', 'sl', 'entry_time_offset', 'percentage_change'])
    if mutation_type == 'tp':
        new_solution['tp'] = np.random.choice(tp_values)
    elif mutation_type == 'sl':
        new_solution['sl'] = np.random.choice(sl_values)    
    elif mutation_type == 'percentage_change':
        new_solution['percentage_change'] = np.random.choice(percentage_change_values)
    return new_solution

# Step 3: Evaluate the solution using the backtest function
def evaluate_solution(signal, solution):
    result = backtest_trades(
        price_data, signal, 
        tp=solution['tp'], 
        sl=solution['sl'], 
        entry_time_offset=0, 
        percentage_change=solution['percentage_change'], 
        open_order_elimination=120, 
        ignore_time_interval_before=1020, 
        ignore_time_interval_after=0
    )
    if result is not None and not result.empty and 'ROI' in result.columns:
        return result['ROI'].iloc[0]
    else:
        return -float('inf')  # Return a very low fitness if the result is invalid

# Step 4: Implement the Simulated Annealing algorithm
def simulated_annealing(signal, initial_temperature, cooling_rate, max_iterations):
    current_solution = create_initial_solution()
    current_fitness = evaluate_solution(signal, current_solution)
    best_solution = current_solution
    best_fitness = current_fitness
    temperature = initial_temperature

    for iteration in range(max_iterations):
        neighbor = generate_neighbor(current_solution)
        neighbor_fitness = evaluate_solution(signal, neighbor)

        if neighbor_fitness > current_fitness:
            current_solution = neighbor
            current_fitness = neighbor_fitness
            if current_fitness > best_fitness:
                best_solution = current_solution
                best_fitness = current_fitness
        else:
            # Accept the worse solution with a probability that decreases over time
            acceptance_probability = np.exp((neighbor_fitness - current_fitness) / temperature)
            if random.random() < acceptance_probability:
                current_solution = neighbor
                current_fitness = neighbor_fitness

        # Decrease the temperature
        temperature *= cooling_rate

        print(f"Iteration {iteration + 1}/{max_iterations}: Current Fitness = {current_fitness:.4f}, Best Fitness = {best_fitness:.4f}, Temperature = {temperature:.4f}")

    return best_solution, best_fitness

# Step 5: Run the optimization for each signal in the signal data
def optimize_parameters_for_each_signal(signal_data):
    optimized_parameters = []

    for index, signal in signal_data.iterrows():
        print(f"\nOptimizing parameters for signal {index + 1}/{len(signal_data)} at {signal['Datetime']}")

        # Run the simulated annealing optimization
        best_solution, best_fitness = simulated_annealing(
            signal,
            initial_temperature=1000,  # Start with a high temperature
            cooling_rate=0.95,  # Reduce temperature by 5% each iteration
            max_iterations=100  # Number of iterations
        )
        
        # If the optimized fitness (ROI) is not zero, save the results
        if best_fitness != 0:
            optimized_parameters.append({
                'Date': signal['Datetime'],
                'Optimized TP': best_solution['tp'],
                'Optimized SL': best_solution['sl'],
                'Percentage Change': best_solution['percentage_change'],
                'ROI': best_fitness
            })

    # Convert the list of dictionaries into a DataFrame
    optimized_df = pd.DataFrame(optimized_parameters)
    
    # Filter out any signals with ROI equal to 0
    optimized_df = optimized_df[optimized_df['ROI'] != 0]

    print(f"\nOptimization completed. Total optimized signals: {len(optimized_df)}")
    return optimized_df


In [ ]:
# Step 1: Filter the signal_data to include only rows where the Signal column is -1 or 1
filtered_signals = signal_data[(signal_data['Signal'] == -1) | (signal_data['Signal'] == 1)]
optimized_params = optimize_parameters_for_each_signal(filtered_signals)

In [ ]:
optimized_df = pd.DataFrame(optimized_params)
optimized_df

In [ ]:
# Implementing the corrected approach
import pandas as pd
import numpy as np

# Load the price data
price_data = pd.read_csv('E:\Signal Backtesting\Input\price_2023-06-01_to_2024-08-4_min.csv', parse_dates=['Datetime'])
combined_df = pd.read_csv('E:\Signal Backtesting\Output\optimized_single_signal_without_timeoffset.csv', parse_dates=['Date'])

# Resample the price data to a 15-minute interval
price_data_15min = price_data.resample('15min', on='Datetime').agg({
    'Open': 'first',
    'High': 'max',
    'Low': 'min',
    'Close': 'last'
}).dropna().reset_index()

# Function to calculate ATR
def calculate_atr(df, window):
    high_low = df['High'] - df['Low']
    high_close = np.abs(df['High'] - df['Close'].shift())
    low_close = np.abs(df['Low'] - df['Close'].shift())
    true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    return true_range.rolling(window=window).mean().iloc[-1]  # Use .iloc[-1] to get the most recent ATR value

# Initialize columns for ATR in the combined_df
for window in range(1, 13):
    combined_df[f'ATR_{window}'] = np.nan

# Calculate ATR for each date in the combined_df
for idx, row in combined_df.iterrows():
    current_date = row['Date']
    
    # Extract historical price data up to the current date
    historical_data = price_data_15min[price_data_15min['Datetime'] <= current_date]
    
    if len(historical_data) > 0:
        # Calculate ATR for windows from 1 to 60
        for window in range(1, 13):
            atr_value = calculate_atr(historical_data, window)
            combined_df.at[idx, f'ATR_{window}'] = atr_value     

# Initialize the columns for (ATR_i / price) * 100 in the combined_df
for i in range(1, 13):
    combined_df[f'ATR_{i}_pct'] = np.nan

# Calculate (ATR_i / price) * 100 for each date in the combined_df
for idx, row in combined_df.iterrows():
    current_date = row['Date']
    
    # Find the corresponding price for the current date
    price_row = price_data_15min[price_data_15min['Datetime'] <= current_date].iloc[-1]
    current_price = price_row['Close']
    
    # Calculate (ATR_i / price) * 100 for i = 1 to 60
    for i in range(1, 13):
        atr_value = row[f'ATR_{i}']
        combined_df.at[idx, f'ATR_{i}_pct'] = (atr_value / current_price) 

# Calculate residuals for (Optimized TP - ATR_i_pct), (Optimized SL - ATR_i_pct), and (Percentage Change - ATR_i_pct)
for i in range(1, 13):
    combined_df[f'TP_ATR_{i}_residual'] = combined_df['Optimized TP'] - combined_df[f'ATR_{i}_pct']
    combined_df[f'SL_ATR_{i}_residual'] = combined_df['Optimized SL'] - combined_df[f'ATR_{i}_pct']
    combined_df[f'PC_ATR_{i}_residual'] = combined_df['Percentage Change'] - combined_df[f'ATR_{i}_pct']


# Save the modified DataFrame with ATR columns
output_path = 'E:\Signal Backtesting\Output\combined_with_atr_analysis_with_12_windows.csv'
combined_df.to_csv(output_path, index=False)

combined_df.head()  # Display the first few rows to confirm the operation

In [ ]:
import plotly.graph_objs as go
from plotly.subplots import make_subplots
# Function to create and save a subplot with 60 rows for a specific residual type
def save_residuals_subplot_as_html(df, residual_type, output_path):
    fig = make_subplots(rows=60, cols=1, shared_xaxes=True, vertical_spacing=0.01,
                        subplot_titles=[f'{residual_type} Residual (Window {i})' for i in range(1, 61)])
    
    for i in range(1, 61):
        fig.add_trace(go.Scatter(x=df['Date'], y=df[f'{residual_type}_ATR_{i}_residual'], 
                                 mode='markers', marker=dict(size=4), name=f'{residual_type}_ATR_{i}_residual'), 
                      row=i, col=1)
    
    fig.update_layout(height=20000, width=1800, title_text=f"All {residual_type} Residuals Over Time", showlegend=False)
    fig.write_html(output_path)

# Save TP residuals subplot as HTML
save_residuals_subplot_as_html(combined_df, 'TP', 'E:\Signal Backtesting\Output\\tp_residuals.html')

# Save SL residuals subplot as HTML
save_residuals_subplot_as_html(combined_df, 'SL', 'E:\Signal Backtesting\Output\\sl_residuals.html')

# Save PC residuals subplot as HTML
save_residuals_subplot_as_html(combined_df, 'PC', 'E:\Signal Backtesting\Output\\pc_residuals.html')

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import sem

df = pd.read_csv('E:\Signal Backtesting\Output\combined_with_atr_analysis_with_12_windows.csv',
                            parse_dates=['Date'])

# Function to calculate the most frequent range
def most_frequent_range(data, bins):
    hist, bin_edges = np.histogram(data, bins=bins)
    max_bin_idx = np.argmax(hist)
    most_freq_range = (bin_edges[max_bin_idx], bin_edges[max_bin_idx + 1])
    return most_freq_range, data[(data >= most_freq_range[0]) & (data < most_freq_range[1])]

# Function to calculate SEM and SEV
def calculate_errors(data, population_size=None):
    n = len(data)
    N = len(residuals) if population_size is None else population_size
    sample_var = np.var(data, ddof=1)
    
    sem_sample = sem(data)  # SEM for the sample
    sem_population = sample_var / np.sqrt(N)  # SEM for the population
    
    sev_sample = np.sqrt(2 * (sample_var ** 2) / (n - 1))  # SEV for the sample
    sev_population = np.sqrt(2 * (sample_var ** 2) / (N - 1))  # SEV for the population
    
    return {
        "sem_sample": sem_sample,
        "sem_population": sem_population,
        "sev_sample": sev_sample,
        "sev_population": sev_population
    }

# Initialize results storage
results = []

# Assuming df is your DataFrame containing the residuals
for column in df.columns:
    if column.endswith("residual"):
        residuals = df[column].dropna().values
        
        # Sturges' Rule
        bins_sturges = int(np.ceil(np.log2(len(residuals))) + 1)
        sturges_range, sturges_data = most_frequent_range(residuals, bins_sturges)
        sturges_errors = calculate_errors(sturges_data)

        # Rice Rule
        bins_rice = int(2 * len(residuals) ** (1/3))
        rice_range, rice_data = most_frequent_range(residuals, bins_rice)
        rice_errors = calculate_errors(rice_data)

        # Square-root Choice
        bins_sqrt = int(np.sqrt(len(residuals)))
        sqrt_range, sqrt_data = most_frequent_range(residuals, bins_sqrt)
        sqrt_errors = calculate_errors(sqrt_data)

        # Freedman-Diaconis Rule
        q25, q75 = np.percentile(residuals, [25, 75])
        bin_width_fd = 2 * (q75 - q25) * len(residuals) ** (-1/3)
        bins_fd = int((residuals.max() - residuals.min()) / bin_width_fd)
        fd_range, fd_data = most_frequent_range(residuals, bins_fd)
        fd_errors = calculate_errors(fd_data)
        
        # Append results
        results.append({
            "Column": column,
            "Method": "Sturges",
            **sturges_errors
        })
        results.append({
            "Column": column,
            "Method": "Rice",
            **rice_errors
        })
        results.append({
            "Column": column,
            "Method": "Square-root",
            **sqrt_errors
        })
        results.append({
            "Column": column,
            "Method": "Freedman-Diaconis",
            **fd_errors
        })

# Convert results to DataFrame
results_df = pd.DataFrame(results)

# Save to CSV
results_df.to_csv('E:\Signal Backtesting\Output\\residuals_standard_errors.csv', index=False)

print("Results saved to 'residuals_standard_errors.csv'")


In [ ]:
# Load the CSV file
df = pd.read_csv('E:\Signal Backtesting\Output\\residuals_standard_errors.csv')

# Group by Method
grouped = df.groupby('Method')

# Calculate the metrics
performance_metrics = grouped.agg({
    'sem_sample': ['mean', 'median', 'std'],
    'sem_population': ['mean', 'median', 'std'],
    'sev_sample': ['mean', 'median', 'std'],
    'sev_population': ['mean', 'median', 'std']
}).reset_index()

# Display the performance metrics
print(performance_metrics)

# Save the performance metrics to a new CSV
performance_metrics.to_csv('method_performance_metrics.csv', index=False)

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objs as go
from plotly.subplots import make_subplots

df = pd.read_csv('E:\Signal Backtesting\Output\combined_with_atr_analysis_with_12_windows.csv')

# Identify all the columns ending with 'residual'
residual_columns = [col for col in df.columns if col.endswith('residual')]

# Initialize a dictionary to store the results
most_frequent_bin_values = {}

# Define function to create subplots for a given category and also calculate values in the most frequent bin
def create_subplots_for_category(category, df):
    category_columns = [col for col in residual_columns if col.startswith(category)]
    num_plots = len(category_columns)
    
    if num_plots == 0:
        print(f"No columns found for category '{category}'. Skipping...")
        return None
    
    # Create subplots layout
    fig = make_subplots(rows=num_plots, cols=1, shared_xaxes=True, 
                        subplot_titles=category_columns)

    for i, column in enumerate(category_columns):
        data = df[column].dropna()  # Drop any NaN values
        num_bins = int(np.sqrt(len(data)))  # Square-root method for bin calculation
        
        # Use pandas cut to create bins
        data_binned = pd.cut(data, bins=num_bins, include_lowest=True)
        
        # Find the most frequent bin
        most_frequent_bin = data_binned.value_counts().idxmax()
        
        # Filter the values that fall within the most frequent bin
        values_in_most_frequent_bin = data[data_binned == most_frequent_bin]
        
        # Store the results
        most_frequent_bin_values[column] = values_in_most_frequent_bin.tolist()

        # Plot the histogram using the same binning
        fig.add_trace(
            go.Histogram(x=data, xbins=dict(start=data.min(), end=data.max(), size=(data.max() - data.min()) / num_bins), 
                         name=column),
            row=i+1, col=1
        )

    # Update layout
    fig.update_layout(title_text=f'Distribution of {category} Residuals', height=200*num_plots)
    return fig

# Define the relevant categories based on the columns found in the DataFrame
categories = ['TP_ATR', 'SL_ATR', 'PC_ATR']

# Create and save plots for each category as HTML
for category in categories:
    fig = create_subplots_for_category(category, df)
    if fig:
        fig.write_html(f"E:\Signal Backtesting\Output\\{category.lower()}_residuals_distribution_consistent.html")
        fig.show()

# Convert the dictionary to a DataFrame for better visualization
most_frequent_bin_df = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in most_frequent_bin_values.items()]))

# Save the results to a CSV file
most_frequent_bin_df.to_csv('E:\Signal Backtesting\Output\most_frequent_bin_values_consistent.csv', index=False)

print("Results saved to 'most_frequent_bin_values_consistent.csv'")

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Load the dataset
df = pd.read_csv('E:\Signal Backtesting\Output\combined_with_atr_analysis_with_192_windows.csv')

# Identify columns that end with '1_residual' or '192_residual'
# Specify the relevant residual columns
residual_columns = [
    'TP_ATR_192_residual', 'SL_ATR_192_residual', 'PC_ATR_192_residual',
    'TP_ATR_1_residual', 'SL_ATR_1_residual', 'PC_ATR_1_residual'
]

# Initialize a dictionary to store the results
most_frequent_bin_values = {}

# Define function to create subplots for a given category and also calculate values in the most frequent bin
def create_subplots_for_category(category, df):
    category_columns = [col for col in residual_columns if col.startswith(category)]
    num_plots = len(category_columns)
    
    if (num_plots == 0):
        print(f"No columns found for category '{category}'. Skipping...")
        return None
    
    # Create subplots layout
    fig = make_subplots(rows=num_plots, cols=1, shared_xaxes=True, 
                        subplot_titles=category_columns)

    for i, column in enumerate(category_columns):
        data = df[column].dropna()  # Drop any NaN values
        num_bins = int(np.sqrt(len(data)))  # Square-root method for bin calculation
        
        # Use pandas cut to create bins
        data_binned = pd.cut(data, bins=num_bins, include_lowest=True)
        
        # Find the most frequent bin
        most_frequent_bin = data_binned.value_counts().idxmax()
        
        # Filter the values that fall within the most frequent bin
        values_in_most_frequent_bin = data[data_binned == most_frequent_bin]
        
        # Store the results
        most_frequent_bin_values[column] = values_in_most_frequent_bin.tolist()

        # Plot the histogram using the same binning
        fig.add_trace(
            go.Histogram(x=data, xbins=dict(start=data.min(), end=data.max(), size=(data.max() - data.min()) / num_bins), 
                         name=column),
            row=i+1, col=1
        )

    # Update layout
    fig.update_layout(title_text=f'Distribution of {category} Residuals', height=200*num_plots)
    return fig

# Define the relevant categories based on the columns found in the DataFrame
categories = ['TP_ATR', 'SL_ATR', 'PC_ATR']

# Create and save plots for each category as HTML
for category in categories:
    fig = create_subplots_for_category(category, df)
    if fig:
        fig.write_html(f"E:\Signal Backtesting\Output\\{category.lower()}_residuals_distribution_consistent_windows=192.html")
        fig.show()

# Convert the dictionary to a DataFrame for better visualization
most_frequent_bin_df = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in most_frequent_bin_values.items()]))

# Save the results to a CSV file
most_frequent_bin_df.to_csv('E:\Signal Backtesting\Output\most_frequent_bin_values_consistent_window=192&1.csv', index=False)

print("Results saved to 'most_frequent_bin_values_consistent.csv'")

In [6]:
import pandas as pd
import numpy as np

# Load the price data
price_data = pd.read_csv('E:\Signal Backtesting\Input\price_2023-06-01_to_2024-08-4_min.csv', parse_dates=['Datetime'])
combined_df = pd.read_csv('E:\Signal Backtesting\Output\optimized_single_signal_without_timeoffset.csv', parse_dates=['Date'])

# Resample the price data to a 15-minute interval
price_data_15min = price_data.resample('15min', on='Datetime').agg({
    'Open': 'first',
    'High': 'max',
    'Low': 'min',
    'Close': 'last'
}).dropna().reset_index()

# Function to calculate ATR
def calculate_atr(df, window):
    high_low = df['High'] - df['Low']
    high_close = np.abs(df['High'] - df['Close'].shift())
    low_close = np.abs(df['Low'] - df['Close'].shift())
    true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    return true_range.rolling(window=window).mean().iloc[-1]  # Most recent ATR value


# Function to calculate the trend
def calculate_trend(df, window):
    return df['Close'].diff(window).iloc[-1] / window  # Use .iloc[-1] to get the most recent trend value

# Initialize columns for trend in the combined_df with windows 2, 4, 6
for window in [2, 4, 6]:
    combined_df[f'Trend_{window}'] = np.nan
    
# Initialize the column for ATR in the combined_df with window 1
combined_df['ATR_1'] = np.nan    

# Calculate trend for each date in the combined_df
for idx, row in combined_df.iterrows():
    current_date = row['Date']
    
    # Extract historical price data up to the current date
    historical_data = price_data_15min[price_data_15min['Datetime'] <= current_date]
    
    if len(historical_data) > 0:
        # Calculate trend for windows 2, 4, 6
        for window in [2, 4, 6]:
            trend_value = calculate_trend(historical_data, window)
            combined_df.at[idx, f'Trend_{window}'] = trend_value
            
        # Calculate ATR for window 1
        atr_value = calculate_atr(historical_data, 1)
        combined_df.at[idx, 'ATR_1'] = atr_value    

# Initialize the columns for (Trend_i / price) in the combined_df
for window in [2, 4, 6]:
    combined_df[f'Trend_{window}_pct'] = np.nan
    
combined_df['ATR_1_pct'] = np.nan    
    
    
# Calculate (Trend_i / price) for each date in the combined_df
for idx, row in combined_df.iterrows():
    current_date = row['Date']
    
    # Find the corresponding price for the current date
    price_row = price_data_15min[price_data_15min['Datetime'] <= current_date].iloc[-1]
    current_price = price_row['Close']
    
    # Calculate (Trend_i / price) for i = 2, 4, 6
    for window in [2, 4, 6]:
        trend_value = row[f'Trend_{window}']
        combined_df.at[idx, f'Trend_{window}_pct'] = trend_value / current_price
        
    # Calculate (ATR_1 / price)
    atr_value = row['ATR_1']
    combined_df.at[idx, 'ATR_1_pct'] = atr_value / current_price    

# Save the modified DataFrame with trend columns
output_path = 'E:\Signal Backtesting\Output\\analysis residuals with 15min interval ATR and 60 windows\combined_with_atr_analysis.csv'
combined_df.to_csv(output_path, index=False)


In [17]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from sklearn.preprocessing import StandardScaler

combined_df = pd.read_csv('E:\Signal Backtesting\Output\\analysis residuals with 15min interval ATR and 60 windows\combined_with_atr_analysis.csv', parse_dates=['Date'])

# Load and standardize the data
price_data = pd.read_csv('E:\Signal Backtesting\Input\price_2023-06-01_to_2024-08-4_min.csv', parse_dates=['Datetime'], index_col='Datetime')

# Define the function to calculate tp, sl, and percentage change based on weights
def calculate_values(weights, df):
    w_tp = weights[0:4]
    w_sl = weights[4:8]
    w_pc = weights[8:12]
    
    df['calculated_tp'] = (w_tp[0] * df['ATR_1_pct'] +
                           w_tp[1] * df['Trend_2_pct'] +
                           w_tp[2] * df['Trend_4_pct'] +
                           w_tp[3] * df['Trend_6_pct'])
    
    df['calculated_sl'] = (w_sl[0] * df['ATR_1_pct'] +
                           w_sl[1] * df['Trend_2_pct'] +
                           w_sl[2] * df['Trend_4_pct'] +
                           w_sl[3] * df['Trend_6_pct'])
    
    df['calculated_percentage_change'] = (w_pc[0] * df['ATR_1_pct'] +
                                          w_pc[1] * df['Trend_2_pct'] +
                                          w_pc[2] * df['Trend_4_pct'] +
                                          w_pc[3] * df['Trend_6_pct'])
    return df

# Define the objective function: the sum of squared differences between real and calculated values
def objective_function(weights, df, alpha=0.01):
    df = calculate_values(weights, df)
    tp_error = np.sum((df['Optimized TP'] - df['calculated_tp']) ** 2)
    sl_error = np.sum((df['Optimized SL'] - df['calculated_sl']) ** 2)
    percentage_change_error = np.sum((df['Percentage Change'] - df['calculated_percentage_change']) ** 2)
    
    # L2 Regularization
    regularization_term = alpha * np.sum(weights ** 2)
    
    return 2*tp_error + sl_error + percentage_change_error + regularization_term

# Initial guess for the weights (12 weights: 4 for tp, 4 for sl, 4 for percentage change)
initial_weights = np.array([0.25, 0.25, 0.25, 0.25,  # weights for tp
                            0.25, 0.25, 0.25, 0.25,  # weights for sl
                            0.25, 0.25, 0.25, 0.25]) # weights for percentage change

# Optimization: Find the weights that minimize the objective function
result = minimize(objective_function, initial_weights, args=(combined_df), method='L-BFGS-B')

# Extract the optimized weights
optimized_weights = result.x
optimized_weights_tp = optimized_weights[0:4]
optimized_weights_sl = optimized_weights[4:8]
optimized_weights_pc = optimized_weights[8:12]

# Apply the optimized weights to calculate the final tp, sl, and percentage change
combined_df = calculate_values(optimized_weights, combined_df)

# Display the optimized weights and first few rows of the DataFrame
print("Optimized Weights for TP:", optimized_weights_tp)
print("Optimized Weights for SL:", optimized_weights_sl)
print("Optimized Weights for Percentage Change:", optimized_weights_pc)

result= combined_df[['calculated_tp', 'Optimized TP', 'calculated_sl', 'Optimized SL', 'calculated_percentage_change', 'Percentage Change']]
result
# # Save the results to a CSV if needed
# output_path = 'E:\Signal Backtesting\Output\combined_with_optimized_values.csv'
# combined_df.to_csv(output_path, index=False)

Optimized Weights for TP: [0.84365478 0.042479   0.02279111 0.0140213 ]
Optimized Weights for SL: [ 0.46237542  0.02992558  0.00801717 -0.00200349]
Optimized Weights for Percentage Change: [ 0.03890126  0.00311409  0.00063127 -0.00066576]


,calculated_tp,Optimized TP,calculated_sl,Optimized SL,calculated_percentage_change,Percentage Change
0,0.000980,0.007,0.000531,0.008,0.000044,0.0003
1,0.001231,0.013,0.000680,0.004,0.000057,0.0003
2,0.001645,0.012,0.000909,0.008,0.000077,0.0002
3,0.002130,0.011,0.001170,0.003,0.000099,0.0001
4,0.001514,0.011,0.000831,0.011,0.000070,0.0004
...,...,...,...,...,...,...
222,0.004382,0.013,0.002409,0.009,0.000202,0.0010
223,0.012123,0.008,0.006641,0.003,0.000560,0.0005
224,0.002230,0.004,0.001218,0.007,0.000102,0.0011
225,0.002108,0.013,0.001150,0.005,0.000097,0.0007


In [20]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

# Prepare the data
X = combined_df[['ATR_1_pct', 'Trend_2_pct', 'Trend_4_pct', 'Trend_6_pct']]

# Polynomial regression with degree 2
poly = PolynomialFeatures(degree=3, include_bias=False)

# Fit models for tp, sl, and percentage change
y_tp = combined_df['Optimized TP']
y_sl = combined_df['Optimized SL']
y_pct = combined_df['Percentage Change']

# Create pipelines with Polynomial Features and Linear Regression
model_tp = make_pipeline(poly, LinearRegression())
model_sl = make_pipeline(poly, LinearRegression())
model_pct = make_pipeline(poly, LinearRegression())

# Fit the models
model_tp.fit(X, y_tp)
model_sl.fit(X, y_sl)
model_pct.fit(X, y_pct)

# Predict using the models
combined_df['calculated_tp_poly'] = model_tp.predict(X)
combined_df['calculated_sl_poly'] = model_sl.predict(X)
combined_df['calculated_pct_poly'] = model_pct.predict(X)

# Extract the coefficients (these are the "weights" in the polynomial model)
weights_tp = model_tp.named_steps['linearregression'].coef_
weights_sl = model_sl.named_steps['linearregression'].coef_
weights_pct = model_pct.named_steps['linearregression'].coef_


calculated_and_real_values = combined_df[['calculated_tp_poly', 'Optimized TP', 
                                 'calculated_sl_poly', 'Optimized SL', 
                                 'calculated_pct_poly', 'Percentage Change']]
# Display the first few rows with the calculated values
print(calculated_and_real_values.to_string())

# weights_tp, weights_sl, weights_pct



     calculated_tp_poly  Optimized TP  calculated_sl_poly  Optimized SL  calculated_pct_poly  Percentage Change
0              0.008816         0.007            0.006420         0.008             0.000357             0.0003
1              0.010596         0.013            0.008507         0.004             0.000693             0.0003
2              0.010140         0.012            0.007421         0.008             0.000592             0.0002
3              0.010212         0.011            0.007854         0.003             0.000647             0.0001
4              0.010613         0.011            0.008623         0.011             0.000748             0.0004
5              0.011077         0.013            0.009239         0.009             0.000731             0.0014
6              0.010791         0.013            0.009490         0.012             0.000634             0.0002
7              0.009313         0.005            0.006606         0.003             0.000842            

In [4]:
from scipy.optimize import minimize
import numpy as np

# Define the function to calculate tp, sl, and percentage change based on weights
def calculate_values(weights, df):
    w_tp = weights[0:4]
    w_sl = weights[4:8]
    w_pc = weights[8:12]
    
    df['calculated_tp'] = (w_tp[0] * df['ATR_1_pct'] +
                           w_tp[1] * df['Trend_2_pct'] +
                           w_tp[2] * df['Trend_4_pct'] +
                           w_tp[3] * df['Trend_6_pct'])
    
    df['calculated_sl'] = (w_sl[0] * df['ATR_1_pct'] +
                           w_sl[1] * df['Trend_2_pct'] +
                           w_sl[2] * df['Trend_4_pct'] +
                           w_sl[3] * df['Trend_6_pct'])
    
    df['calculated_percentage_change'] = (w_pc[0] * df['ATR_1_pct'] +
                                          w_pc[1] * df['Trend_2_pct'] +
                                          w_pc[2] * df['Trend_4_pct'] +
                                          w_pc[3] * df['Trend_6_pct'])
    return df

# Define the objective function: the sum of squared differences between real and calculated values
def objective_function(weights, df):
    df = calculate_values(weights, df)
    tp_error = np.sum((df['Optimized TP'] - df['calculated_tp']) ** 2)
    sl_error = np.sum((df['Optimized SL'] - df['calculated_sl']) ** 2)
    percentage_change_error = np.sum((df['Percentage Change'] - df['calculated_percentage_change']) ** 2)
    return tp_error + sl_error + percentage_change_error

# Initial guess for the weights (12 weights: 4 for tp, 4 for sl, 4 for percentage change)
initial_weights = np.array([0.25, 0.25, 0.25, 0.25,  # weights for tp
                            0.25, 0.25, 0.25, 0.25,  # weights for sl
                            0.25, 0.25, 0.25, 0.25]) # weights for percentage change

# Optimization: Find the weights that minimize the objective function using Nelder-Mead
result = minimize(objective_function, initial_weights, args=(combined_df), method='Nelder-Mead')

# Extract the optimized weights
optimized_weights = result.x
optimized_weights_tp = optimized_weights[0:4]
optimized_weights_sl = optimized_weights[4:8]
optimized_weights_pc = optimized_weights[8:12]

# Apply the optimized weights to calculate the final tp, sl, and percentage change
combined_df = calculate_values(optimized_weights, combined_df)

# Print the results
print("Optimized Weights (TP):", optimized_weights_tp)
print("Optimized Weights (SL):", optimized_weights_sl)
print("Optimized Weights (Percentage Change):", optimized_weights_pc)

Optimized Weights (TP): [1.5391358  0.0308585  0.61025374 0.45786771]
Optimized Weights (SL): [1.23235548 0.16298034 0.73317409 0.04343284]
Optimized Weights (Percentage Change): [ 0.11784056 -0.28402733  0.47535977  0.11242254]


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

# Load the dataset
df = pd.read_csv('E:\Signal Backtesting\Output\\analysis residuals with 15min interval ATR and 60 windows\combined_with_atr_analysis.csv', parse_dates=['Date'])

# Select only the desired features
X = df[['ATR_1_pct', 'Trend_2_pct', 'Trend_4_pct', 'Trend_6_pct']]

# Define the target variables
y_tp = df['Optimized TP']
y_sl = df['Optimized SL']
y_pc = df['Percentage Change']


# Splitting the data into training and test sets
X_train, X_test, y_tp_train, y_tp_test = train_test_split(X, y_tp, test_size=0.2, random_state=42)
X_train, X_test, y_sl_train, y_sl_test = train_test_split(X, y_sl, test_size=0.2, random_state=42)
X_train, X_test, y_pc_train, y_pc_test = train_test_split(X, y_pc, test_size=0.2, random_state=42)

# Initialize decision tree models with regularization
tree_tp = DecisionTreeRegressor(max_depth=3, min_samples_split=15, random_state=42)
tree_sl = DecisionTreeRegressor(max_depth=3, min_samples_split=15, random_state=42)
tree_pc = DecisionTreeRegressor(max_depth=3, min_samples_split=15, random_state=42)

# Train the models
tree_tp.fit(X_train, y_tp_train)
tree_sl.fit(X_train, y_sl_train)
tree_pc.fit(X_train, y_pc_train)

# Make predictions
y_tp_pred = tree_tp.predict(X_test)
y_sl_pred = tree_sl.predict(X_test)
y_pc_pred = tree_pc.predict(X_test)

# Calculate Mean Squared Error
mse_tp = mean_squared_error(y_tp_test, y_tp_pred)
mse_sl = mean_squared_error(y_sl_test, y_sl_pred)
mse_pc = mean_squared_error(y_pc_test, y_pc_pred)

print("MSE TP:", mse_tp)
print("MSE SL:", mse_sl)
print("MSE Percentage Change:", mse_pc)

# Extract and display feature importance
tp_feature_importance = tree_tp.feature_importances_
sl_feature_importance = tree_sl.feature_importances_
pc_feature_importance = tree_pc.feature_importances_

# Combine feature names with their importance scores
feature_names = X.columns
tp_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': tp_feature_importance})
sl_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': sl_feature_importance})
pc_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': pc_feature_importance})

# Sort the features by importance
tp_importance_df = tp_importance_df.sort_values(by='Importance', ascending=False)
sl_importance_df = sl_importance_df.sort_values(by='Importance', ascending=False)
pc_importance_df = pc_importance_df.sort_values(by='Importance', ascending=False)

print("\nFeature Importance for TP:")
print(tp_importance_df)
print("\nFeature Importance for SL:")
print(sl_importance_df)
print("\nFeature Importance for Percentage Change:")
print(pc_importance_df)

# Create a DataFrame to compare real and predicted values
comparison_df = pd.DataFrame({
    'Real TP': y_tp_test,
    'Predicted TP': y_tp_pred,
    'Real SL': y_sl_test,
    'Predicted SL': y_sl_pred,
    'Real Percentage Change': y_pc_test,
    'Predicted Percentage Change': y_pc_pred
})

# Display the first few rows of the comparison
print("\nComparison of Real vs Predicted Values:")
print(comparison_df.head(50))  # Display the first 10 rows for a quick comparison

# Save the comparison to a CSV file
comparison_df.to_csv('comparison_real_vs_predicted_corrected.csv', index=False)


MSE TP: 1.1323906710118689e-05
MSE SL: 9.88880368914376e-06
MSE Percentage Change: 1.2624119574542522e-07

Feature Importance for TP:
       Feature  Importance
2  Trend_4_pct    0.748066
1  Trend_2_pct    0.251934
0    ATR_1_pct    0.000000
3  Trend_6_pct    0.000000

Feature Importance for SL:
       Feature  Importance
2  Trend_4_pct    0.854182
0    ATR_1_pct    0.145818
1  Trend_2_pct    0.000000
3  Trend_6_pct    0.000000

Feature Importance for Percentage Change:
       Feature  Importance
2  Trend_4_pct    0.380292
3  Trend_6_pct    0.333503
0    ATR_1_pct    0.154072
1  Trend_2_pct    0.132134

Comparison of Real vs Predicted Values:
     Real TP  Predicted TP  Real SL  Predicted SL  Real Percentage Change  \
9      0.010      0.010481    0.009      0.007714                  0.0007   
143    0.008      0.010481    0.013      0.008309                  0.0011   
15     0.013      0.010481    0.009      0.005857                  0.0014   
124    0.013      0.010481    0.007      

In [4]:
import pandas as pd

# Load the new dataset
new_df = pd.read_csv('E:\Signal Backtesting\Output\\test_dataset_with_date.csv')
# Select the same features used during training
X_new = new_df[['ATR_1_pct', 'Trend_2_pct', 'Trend_4_pct', 'Trend_6_pct']]

# Assuming the trained models are `tree_tp`, `tree_sl`, and `tree_pc`
new_tp_pred = tree_tp.predict(X_new)
new_sl_pred = tree_sl.predict(X_new)
new_pc_pred = tree_pc.predict(X_new)

# Add the predictions to the original dataset
new_df['Predicted TP'] = new_tp_pred
new_df['Predicted SL'] = new_sl_pred
new_df['Predicted Percentage Change'] = new_pc_pred

# Assuming the real values are stored in the dataset under the columns 'Optimized TP', 'Optimized SL', and 'Percentage Change'
new_df['TP Difference'] = new_df['Optimized TP'] - new_df['Predicted TP']
new_df['SL Difference'] = new_df['Optimized SL'] - new_df['Predicted SL']
new_df['Percentage Change Difference'] = new_df['Percentage Change'] - new_df['Predicted Percentage Change']

# Save the updated DataFrame to a new CSV file
new_df.to_csv('E:\Signal Backtesting\Output\\new_dataset_with_predictions_and_differences.csv', index=False)

# Optionally, display the first few rows of the updated DataFrame
print(new_df)


               Date  ATR_1_pct  Trend_2_pct  Trend_4_pct  Trend_6_pct  \
0   1/11/2024 17:00   0.007916     0.000483     0.000888    -0.003475   
1   5/18/2024 18:00   0.001932    -0.000246    -0.000560    -0.000214   
2   1/15/2024 23:00   0.002602    -0.002290     0.000540    -0.000865   
3    4/30/2024 4:00   0.001435     0.000229     0.000258    -0.000404   
4    5/25/2024 9:00   0.001345     0.000302     0.000348     0.000477   
5   3/21/2024 19:00   0.004116     0.002201    -0.000119    -0.002030   
6    5/4/2024 12:00   0.004267    -0.001991    -0.001175     0.000140   
7     4/7/2024 2:00   0.004649     0.000461     0.001005     0.000950   
8   2/19/2024 13:00   0.002594    -0.000175     0.000277     0.000043   
9   1/23/2024 17:00   0.005502     0.002194     0.002148     0.000088   
10   4/14/2024 1:00   0.008779    -0.004598    -0.001464    -0.002189   
11   6/17/2024 5:00   0.001769     0.001068     0.000957     0.000446   
12  7/13/2024 17:00   0.000945    -0.000531    -0.0

In [26]:
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import pandas as pd

# Load the provided datasets
df_atr = pd.read_csv('E:\Signal Backtesting\Output\\updated_combined_with_atr_analysis.csv')
df_optimized = pd.read_csv('E:\Signal Backtesting\Output\\updated_optimized_weights_rev.csv')

# Create subplots
fig = make_subplots(rows=3, cols=1, subplot_titles=(
    'Comparison of TP Differences',
    'Comparison of SL Differences',
    'Comparison of Percentage Change Differences'
))

# Scatter plot for tp_atr_difference vs tp_difference
fig.add_trace(
    go.Scatter(x=df_atr.index, y=df_atr['tp_atr_difference'], mode='markers', name='TP ATR Difference', marker=dict(color='blue', size = 4)),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=df_optimized.index, y=df_optimized['tp_difference_rev'], mode='markers', name='TP Difference', marker=dict(color='red', size = 4)),
    row=1, col=1
)

# Scatter plot for sl_atr_difference vs sl_difference
fig.add_trace(
    go.Scatter(x=df_atr.index, y=df_atr['sl_atr_difference'], mode='markers', name='SL ATR Difference', marker=dict(color='blue')),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=df_optimized.index, y=df_optimized['sl_difference_rev'], mode='markers', name='SL Difference', marker=dict(color='red')),
    row=2, col=1
)

# Scatter plot for percentage_change_atr_difference vs percentage_change_difference
fig.add_trace(
    go.Scatter(x=df_atr.index, y=df_atr['percentage_change_atr_difference'], mode='markers', name='Percentage Change ATR Difference', marker=dict(color='blue')),
    row=3, col=1
)
fig.add_trace(
    go.Scatter(x=df_optimized.index, y=df_optimized['percentage_change_difference_rev'], mode='markers', name='Percentage Change Difference', marker=dict(color='red')),
    row=3, col=1
)

# Update layout
fig.update_layout(height=1000, width=1200,title_text="Comparisons of Differences")

# Show plot
fig.show()

In [8]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Load the datasets
df_predictions  = pd.read_csv('E:\Signal Backtesting\Output\Optimized weights with regularization.csv')
df_optimized = pd.read_csv('E:\Signal Backtesting\Output\\new_dataset_with_predictions_and_differences.csv')

# Set the Date column as index
df_predictions.set_index('Date', inplace=True)
df_optimized.set_index('Date', inplace=True)

# Find common dates
common_dates = df_predictions.index.intersection(df_optimized.index)

# Filter the data to only include common dates
df_predictions_common = df_predictions.loc[common_dates]
df_optimized_common = df_optimized.loc[common_dates]

# Create subplots
fig = make_subplots(rows=3, cols=1, subplot_titles=(
    'Comparison of TP',
    'Comparison of SL',
    'Comparison of Percentage Change'
))

# Plot for TP
fig.add_trace(
    go.Scatter(x=df_predictions_common.index, y=df_predictions_common['Optimized TP'], mode='markers', name='Optimized TP', marker=dict(color='blue', size=6)),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=df_predictions_common.index, y=df_predictions_common['calculated_tp'], mode='markers', name='Calculated TP', marker=dict(color='red', size=6)),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=df_predictions_common.index, y=df_optimized_common['Predicted TP'], mode='markers', name='Predicted TP', marker=dict(color='green', size=6)),
    row=1, col=1
)

# Plot for SL
fig.add_trace(
    go.Scatter(x=df_predictions_common.index, y=df_predictions_common['Optimized SL'], mode='markers', name='Optimized SL', marker=dict(color='blue', size=6)),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=df_predictions_common.index, y=df_predictions_common['calculated_sl'], mode='markers', name='Calculated SL', marker=dict(color='red', size=6)),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=df_predictions_common.index, y=df_optimized_common['Predicted SL'], mode='markers', name='Predicted SL', marker=dict(color='green', size=6)),
    row=2, col=1
)

# Plot for Percentage Change
fig.add_trace(
    go.Scatter(x=df_predictions_common.index, y=df_predictions_common['Percentage Change'], mode='markers', name='Percentage Change', marker=dict(color='blue', size=6)),
    row=3, col=1
)
fig.add_trace(
    go.Scatter(x=df_predictions_common.index, y=df_predictions_common['calculated_percentage_change'], mode='markers', name='Calculated Percentage Change', marker=dict(color='red', size=6)),
    row=3, col=1
)
fig.add_trace(
    go.Scatter(x=df_predictions_common.index, y=df_optimized_common['Predicted Percentage Change'], mode='markers', name='Predicted Percentage Change', marker=dict(color='green', size=6)),
    row=3, col=1
)

# Update layout
fig.update_layout(height=1500, width=1200 ,title_text="Comparisons of TP, SL, and Percentage Change")

# Show plot
fig.show()